# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya – Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR² dataset defined by a Croissant schema using the `mlcroissant` library.

### Dataset Source
This analysis loads the dataset via its Croissant schema URL and explores its structured record sets and fields using their `@id` references.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their fields by `@id`.

In [ ]:
# List all record sets by @id
record_sets = list(dataset.record_sets)
print("Available record sets (@id):")
for rs in record_sets:
    print(f"  {rs['@id']} - {rs.get('name', '[no name]')}")

# For each record set, list its fields by @id
for rs in record_sets:
    print(f"\nFields for record set @id: {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        # Fields can be references by @id or fully described dicts
        if isinstance(field, dict):
            print(f"  {field.get('@id', '[unknown id]')} - {field.get('name', '[no name]')}")
        else:
            print(f"  {field}")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. Use their `@id`s as shown above.

In [ ]:
# Collect list of record set @id's for extraction
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Fetch records from this record set by its @id
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for record set @id: {record_set_id} with shape {df.shape}")

if record_set_ids:
    # Show first DataFrame columns and a preview
    main_record_set_id = record_set_ids[0]
    print(f"\nColumns for {main_record_set_id}:")
    print(list(dataframes[main_record_set_id].columns))
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering numeric fields, normalization, and grouping. Use field and record set `@id`s.

In [ ]:
# Choose a record set and numeric field for EDA; modify these as needed for your dataset
# First, print available fields to help select a numeric field for demonstration
if record_set_ids:
    record_set_id = main_record_set_id
    df = dataframes[record_set_id]
    print("Available fields (columns):")
    print(list(df.columns))

    # Attempt to detect a numeric field (e.g., log likelihood, coefficient, or similar)
    # Adjust if needed: replace with actual numeric column name by its @id
    numeric_candidates = [col for col in df.columns if df[col].dtype in ['float64', 'int64']]
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
    else:
        # Try a likely candidate by name fragment
        numeric_field = None
        for col in df.columns:
            if any(x in col.lower() for x in ['loglikelihood', 'coef', 'value', 'score', 'iteration']):
                numeric_field = col
                break
        if numeric_field is None:
            print("No obvious numeric field found. Please adjust the field selection below.")

    if numeric_field:
        print(f"\nUsing numeric field: {numeric_field} for EDA.")

        threshold = df[numeric_field].quantile(0.75)  # Use 75th percentile as an example threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.4f}:")
        print(filtered_df.head())

        # Normalize numeric column in filtered set
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            filtered_df[numeric_field].std()
        )
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt to group by a categorical field, e.g., 'ward', 'region', or similar. Pick best candidate.
        group_field = None
        for col in df.columns:
            if df[col].dtype == object and df[col].nunique() < 10:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"\nGrouped means of {numeric_field} by {group_field}:")
            print(grouped_df.head())
        else:
            print("\nNo suitable grouping field found.")
    else:
        print("Please specify a valid numeric field for EDA.")

## 5. Visualization
Visualize the distribution of a numeric field, or relationship between fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If a grouping field exists from above, visualize group means
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(7, 4))
        mean_by_group = df.groupby(group_field)[numeric_field].mean().sort_values()
        mean_by_group.plot(kind='bar')
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrated loading and exploring the FAIR² dataset using `mlcroissant`, focusing on referencing dataset elements by their `@id` fields. We examined metadata, listed record sets and fields, extracted tabular data, conducted basic EDA including numeric filtering and normalization, and created simple visualizations. You can further extend the analysis by referencing additional `@id`s found in the Croissant schema and adapting the workflow to the dataset's structure.